# Training & Inference Details

Master the numerical foundations of LLM training: loss functions, optimizers, normalization, precision, and sampling strategies. Understand why certain architectural choices matter for performance and stability.

[Open this lesson on the site](https://llm.thelittleone.rocks/#/module/training-inference-details)


## Runtime setup

Before running, set **Runtime -> Change runtime type -> T4 GPU** (or any available GPU). Colab provides PyTorch with CUDA support, so these notebooks run on a real GPU rather than Apple's MPS backend. If a code sample uses `device = torch.device('mps' ...)`, it will fall back to `cpu` on Colab unless you adapt it; replace `'mps'` with `'cuda'` for GPU execution.


In [ ]:
!pip install torch numpy

---

## Lesson examples (optional)

These are the runnable code samples from the lesson sections. Run them to experiment with the concepts before tackling the exercise below.


### Lesson example: Cross-entropy Loss for Language Models


In [ ]:
import torch
import torch.nn.functional as F

# Simulate: B=2 batch, T=4 sequence length, V=10000 vocab
B, T, V = 2, 4, 10000
logits = torch.randn(B, T, V)  # Model output
labels = torch.randint(0, V, (B, T))  # True next tokens

# ❌ WRONG: F.cross_entropy(F.softmax(logits, dim=-1), labels)
# ❌ WRONG: probs = F.softmax(logits, dim=-1); F.cross_entropy(probs, labels)

# ✓ CORRECT: Flatten for cross_entropy
logits_flat = logits.view(B * T, V)
labels_flat = labels.view(B * T)
loss = F.cross_entropy(logits_flat, labels_flat)
print(f"Loss: {loss.item():.4f}")

# With label smoothing (built-in)
loss_fn = torch.nn.CrossEntropyLoss(label_smoothing=0.1)
loss_smooth = loss_fn(logits_flat, labels_flat)

# Per-token loss (for weighted average or per-position analysis)
loss_per_token = F.cross_entropy(logits_flat, labels_flat, reduction='none')
print(f"Loss shape: {loss_per_token.shape}")  # (B*T,)
print(f"Mean: {loss_per_token.mean().item():.4f}")

# Example: Downweight padding tokens (assume 0 is padding index)
# weights = (labels_flat != 0).float()
# weighted_loss = (loss_per_token * weights).sum() / weights.sum()


### Lesson example: Optimizers: SGD, Adam, AdamW


In [ ]:
import torch
import torch.optim as optim

model = torch.nn.Linear(10, 5)

# AdamW with weight decay (recommended for LLMs)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.1)

# Parameter groups: decay weights, no decay on bias/norm
decay = []
no_decay = []
for name, param in model.named_parameters():
    if 'weight' in name and param.dim() >= 2:
        decay.append(param)
    else:
        no_decay.append(param)

optimizer = optim.AdamW([
    {'params': decay, 'weight_decay': 0.1},
    {'params': no_decay, 'weight_decay': 0.0}
], lr=1e-3)

# Warmup + Cosine schedule
def get_scheduler(optimizer, warmup_steps, total_steps):
    def lr_lambda(current_step):
        if current_step < warmup_steps:
            return float(current_step) / float(max(1, warmup_steps))
        return max(0.0, float(total_steps - current_step) / float(max(1, total_steps - warmup_steps)))
    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

total_steps = 1000
warmup_steps = 100
scheduler = get_scheduler(optimizer, warmup_steps, total_steps)

# Training loop
for step in range(total_steps):
    loss = torch.randn(1, requires_grad=True)  # Dummy loss
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()
    if step % 200 == 0:
        print(f"Step {step}, LR: {optimizer.param_groups[0]['lr']:.6f}")

# Inspect optimizer state
print(f"Adam running averages for first param: {optimizer.state[model.weight]}")


### Lesson example: LayerNorm, Residuals, GELU


In [ ]:
import torch
import torch.nn as nn

B, T, D = 2, 4, 64  # Batch, sequence length, embedding dim

x = torch.randn(B, T, D)

# LayerNorm: normalize over feature dimension
ln = nn.LayerNorm(D)
x_norm = ln(x)
print(f"Input mean: {x.mean(dim=-1)[0, 0].item():.4f}")  # Variable
print(f"After LayerNorm mean: {x_norm.mean(dim=-1)[0, 0].item():.6f}")  # ~0
print(f"After LayerNorm std: {x_norm.std(dim=-1)[0, 0].item():.4f}")  # ~1

# Residual connection
def residual_block(x, fn):
    return x + fn(x)

# Pre-norm (modern convention)
def transformer_block(x, attn_fn, ffn_fn):
    ln1 = nn.LayerNorm(D)
    ln2 = nn.LayerNorm(D)
    x = x + attn_fn(ln1(x))
    x = x + ffn_fn(ln2(x))
    return x

# GELU activation
gelu = nn.GELU(approximate='tanh')  # 'tanh' for efficiency, 'none' for exact
x_gelu = gelu(x)
print(f"GELU allows gradients through negative values (smooth)")

# Combining all three
class ModernBlock(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, num_heads=4, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        # Pre-norm + Residual
        x = x + self.attn(self.ln1(x), self.ln1(x), self.ln1(x))[0]
        x = x + self.ffn(self.ln2(x))
        return x

block = ModernBlock(D, d_ff=256)
output = block(x)
print(f"Output shape: {output.shape}")


### Lesson example: Mixed Precision on MPS


In [ ]:
import torch
import torch.nn as nn

# Model setup
model = nn.TransformerEncoderLayer(
    d_model=512, nhead=8, dim_feedforward=2048, batch_first=True
)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

x = torch.randn(4, 10, 512)  # (batch, seq_len, embedding_dim)
target = torch.randn(4, 10, 512)

# ✓ Mixed precision with autocast (recommended for MPS)
scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
model = model.to(device)
x, target = x.to(device), target.to(device)

# Training step with autocast
with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
    output = model(x)
    loss = nn.MSELoss()(output, target)

optimizer.zero_grad()
loss.backward()
optimizer.step()

print(f"Loss (bf16): {loss.item():.4f}")
print(f"Master weights dtype: {next(model.parameters()).dtype}")  # Should be fp32

# Explicit bf16 casting for MPS
with torch.autocast('mps', dtype=torch.bfloat16):
    output = model(x)
    loss = nn.MSELoss()(output, target)

# Manual mixed precision (if needed for custom logic)
model = model.to(torch.float32)  # Keep master weights fp32
x_bf16 = x.to(torch.bfloat16)
with torch.no_grad():
    output = model(x_bf16.to(torch.float32))
    # Explicit cast where needed
print("Manual mixed precision for edge cases")

# Check memory savings
print(f"fp32 size: {sum(p.numel() for p in model.parameters()) * 4 / 1e6:.2f} MB")
print(f"bf16 equivalent: {sum(p.numel() for p in model.parameters()) * 2 / 1e6:.2f} MB")


### Lesson example: Sampling: Temperature, Top-k, Top-p


In [ ]:
import torch
import torch.nn.functional as F

def sample_next_token(logits, temperature=1.0, top_k=None, top_p=None):
    """
    Sample next token with temperature, top-k, and top-p.

    Args:
        logits: (vocab_size,) or (batch_size, vocab_size)
        temperature: float, >0
        top_k: int or None (no filtering)
        top_p: float in [0, 1] or None (no filtering)

    Returns:
        token indices
    """
    # Apply temperature
    logits = logits / temperature

    # Top-k filtering
    if top_k is not None:
        top_k = min(top_k, logits.size(-1))
        top_logits, top_indices = torch.topk(logits, top_k, dim=-1)
        # Zero out everything not in top-k
        min_val = top_logits[:, -1] if logits.dim() > 1 else top_logits[-1]
        logits = torch.where(logits >= min_val.unsqueeze(-1), logits, torch.full_like(logits, -1e9))

    # Top-p (nucleus) filtering
    if top_p is not None:
        sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)
        cumsum = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
        # Find cutoff: where cumsum first exceeds top_p
        sorted_logits[cumsum > top_p] = -1e9
        # Unsort back to original vocab order
        logits = torch.scatter(torch.full_like(logits, -1e9), -1, sorted_indices, sorted_logits)

    # Sample from distribution
    probs = F.softmax(logits, dim=-1)
    token = torch.multinomial(probs, num_samples=1)
    return token

# Example: Generate sequence
vocab_size = 10000
max_len = 20
context = [42]  # Start token

for _ in range(max_len):
    # Dummy logits from model
    logits = torch.randn(vocab_size)

    # Greedy
    greedy_token = logits.argmax().item()

    # With temperature
    temp_token = sample_next_token(logits, temperature=0.7)

    # Top-p (nucleus) sampling
    nucleus_token = sample_next_token(logits, temperature=0.8, top_p=0.9)

    # Top-k + temperature
    topk_token = sample_next_token(logits, temperature=0.7, top_k=50)

print(f"Greedy: {greedy_token}, Nucleus: {nucleus_token.item()}, Top-k: {topk_token.item()}")

# Batch version
batch_logits = torch.randn(4, vocab_size)  # 4 sequences
tokens = sample_next_token(batch_logits, temperature=0.8, top_p=0.9)
print(f"Batch tokens shape: {tokens.shape}")


---

## Exercise: Implement Top-p Sampling


Write a function that implements nucleus (top-p) sampling with temperature. Given logits, return a sampled token index. Handle batch inputs and edge cases (p=1.0, p=0.0). Test with different temperature and p values to verify behavior.


In [ ]:
import torch
import torch.nn.functional as F

def top_p_sample(logits, p=0.9, temperature=1.0):
    """
    Sample from logits using nucleus (top-p) sampling.

    Args:
        logits: tensor of shape (vocab_size,) or (batch, vocab_size)
        p: float in [0, 1], cumulative probability threshold
        temperature: float > 0, controls randomness

    Returns:
        sampled token indices of shape () or (batch,)
    """
    # TODO: Implement top-p sampling
    pass

# Test cases
if __name__ == '__main__':
    # Test 1: Simple case
    logits = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
    token = top_p_sample(logits, p=0.9, temperature=1.0)
    print(f"Sampled token: {token}")

    # Test 2: Batch
    batch_logits = torch.randn(4, 10000)
    tokens = top_p_sample(batch_logits, p=0.95, temperature=0.7)
    print(f"Batch tokens shape: {tokens.shape}")

    # Test 3: Edge cases
    tokens_p1 = top_p_sample(logits, p=1.0)  # Should sample from all
    tokens_p0 = top_p_sample(logits, p=0.0)  # Should pick argmax or near-argmax
    print(f"p=1.0 allows full dist, p=0.0 is greedy")

---

When you're done, head back to the [lesson page](https://llm.thelittleone.rocks/#/module/training-inference-details) for the solution and discussion.
